# Strombedarf-Prognose: Startnotebook

**Frage:** Wie viel Strom braucht Deutschland morgen, Stunde für Stunde?

**Daten:**
- Stromverbrauch (Netzlast) je Stunde: Bundesnetzagentur | SMARD.de, Lizenz CC BY 4.0
- Lufttemperatur je Stunde: Deutscher Wetterdienst, abgerufen über die Bright-Sky-API

Das Notebook lädt die Daten, baut einfache Features und vergleicht ein Modell mit einer naiven Prognose. Es ist der Startpunkt für Ihr Team-Projekt.

In [ ]:
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

## Einstellungen

In [ ]:
WOCHEN = 104          # so viele Wochen Stundenwerte laden (gut zwei Jahre)
TEST_WOCHEN = 8       # die letzten Wochen sind Testdaten
STAEDTE = {           # Temperatur: Mittel aus fünf Städten
    "Hamburg": (53.55, 9.99),
    "Berlin": (52.52, 13.40),
    "Köln": (50.94, 6.96),
    "Frankfurt": (50.11, 8.68),
    "München": (48.14, 11.58),
}

## Netzlast von SMARD laden

SMARD liefert die Stundenwerte in Paketen zu je einer Woche. Die Liste der Pakete steht in `index_hour.json`. Filter 410 ist der Stromverbrauch (Netzlast) in ganz Deutschland.

In [ ]:
BASIS = "https://www.smard.de/app/chart_data/410/DE"
pakete = requests.get(f"{BASIS}/index_hour.json", timeout=30).json()["timestamps"]

teile = []
for ts in pakete[-WOCHEN:]:
    serie = requests.get(f"{BASIS}/410_DE_hour_{ts}.json", timeout=30).json()["series"]
    teile.append(pd.DataFrame(serie, columns=["zeit", "last_mwh"]))

last = pd.concat(teile)
last["zeit"] = pd.to_datetime(last["zeit"], unit="ms", utc=True)
last = last.dropna().drop_duplicates("zeit").set_index("zeit").sort_index()
print(len(last), "Stunden von", last.index.min(), "bis", last.index.max())

## Temperatur vom Deutschen Wetterdienst laden

Die Bright-Sky-API fragen wir in Stücken von 180 Tagen ab. Zwei Jahre am Stück lehnt der Server manchmal mit einem Fehler ab.

In [ ]:
von = last.index.min().strftime("%Y-%m-%d")
bis = (last.index.max() + pd.DateOffset(days=1)).strftime("%Y-%m-%d")
grenzen = [d.strftime("%Y-%m-%d") for d in pd.date_range(von, bis, freq="180D") if d < pd.Timestamp(bis)] + [bis]

temps = []
for stadt, (lat, lon) in STAEDTE.items():
    stuecke = []
    for anfang, ende in zip(grenzen[:-1], grenzen[1:]):
        antwort = requests.get(
            "https://api.brightsky.dev/weather",
            params={"lat": lat, "lon": lon, "date": anfang, "last_date": ende},
            timeout=120,
        )
        antwort.raise_for_status()
        stuecke.append(pd.DataFrame(antwort.json()["weather"])[["timestamp", "temperature"]])
    t = pd.concat(stuecke)
    t["zeit"] = pd.to_datetime(t["timestamp"], utc=True)
    temps.append(t.drop_duplicates("zeit").set_index("zeit")["temperature"].rename(stadt))

temperatur = pd.concat(temps, axis=1).mean(axis=1).rename("temperatur")
print(len(temperatur), "Stunden Temperatur")

## Features bauen

Wichtig: Nur Werte verwenden, die man am Vortag schon kennt. Darum nehmen wir die Last von vor zwei Tagen und von vor einer Woche zur selben Stunde, nicht die von gestern.

In [ ]:
df = last.join(temperatur, how="inner")
lokal = df.index.tz_convert("Europe/Berlin")
df["stunde"] = lokal.hour
df["wochentag"] = lokal.dayofweek
df["monat"] = lokal.month
df["last_vor_2_tagen"] = df["last_mwh"].shift(48)
df["last_vor_1_woche"] = df["last_mwh"].shift(168)
df = df.dropna()

features = ["stunde", "wochentag", "monat", "temperatur", "last_vor_2_tagen", "last_vor_1_woche"]
df.tail()

## Zeitlich aufteilen

Die letzten Wochen sind Testdaten. Ein zufälliger Split würde die Zukunft ins Training mischen.

In [ ]:
grenze = df.index.max() - pd.DateOffset(weeks=TEST_WOCHEN)
train, test = df[df.index <= grenze], df[df.index > grenze]
print("Training:", len(train), "Stunden | Test:", len(test), "Stunden ab", grenze)

## Modell und naive Prognose

In [ ]:
model = HistGradientBoostingRegressor(random_state=42)
model.fit(train[features], train["last_mwh"])
pred = model.predict(test[features])

naiv = test["last_vor_1_woche"]  # naive Prognose: wie vor einer Woche zur selben Stunde

print(f"Naiv:   MAPE {mean_absolute_percentage_error(test['last_mwh'], naiv):.2%}  RMSE {root_mean_squared_error(test['last_mwh'], naiv):,.0f} MWh")
print(f"Modell: MAPE {mean_absolute_percentage_error(test['last_mwh'], pred):.2%}  RMSE {root_mean_squared_error(test['last_mwh'], pred):,.0f} MWh")

## Fragen für Ihr Team

- Wer nutzt die Prognose, und wann muss sie fertig sein?
- Feiertage fehlen noch. Wie viel Fehler steckt in Weihnachten und Ostern?
- SMARD und der Wetterdienst liefern laufend neue Daten. Wie oft trainieren Sie neu?